# Skinsight — acne detection model

Fine-tunes a **Hugging Face Vision Transformer** on ACNE04, measures it, and
compares it head-to-head against a **public pretrained acne model** on the
same test split.

**This notebook deploys nothing.** FR-AI-010: a trained model serves a task
only where a recorded evaluation meets the NFR-SAFE-008 thresholds. The final
cells print that verdict plus the Appendix I register entry.

Before running:
1. Obtain **ACNE04** from its authors; confirm the licence permits your use (CON-007).
2. Upload it to Google Drive.
3. **Runtime → Change runtime type → T4 GPU.**
4. Set `DATA_ROOT` below, then **Runtime → Run all** (~20–35 min).

In [ ]:
#@title Setup
!pip -q install transformers onnx onnxruntime scikit-image 2>/dev/null

import hashlib, json, os, random, re, time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from transformers import AutoImageProcessor, AutoModelForImageClassification

print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > Change runtime type > T4 GPU')

In [ ]:
#@title Config — the only cell you must edit
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/skinsight/acne04'  #@param {type:"string"}
OUT_DIR   = '/content/drive/MyDrive/skinsight/output'  #@param {type:"string"}

# OPTION A: a Hugging Face backbone, fine-tuned on ACNE04.
# ViT pretrained on ImageNet-21k. Apache-2.0, provenance documented by Google,
# which is what lets it go in Appendix I -- unlike the acne models below, whose
# training data is undisclosed.
BACKBONE_ID = 'google/vit-base-patch16-224-in21k'  #@param {type:"string"}

# OPTION B: a public acne model, evaluated on the SAME split for comparison.
# MIT licensed. Runs locally, so no dataset images leave this machine.
BASELINE_ID = 'imfarzanansari/skintelligent-acne'  #@param {type:"string"}

MODEL_ID, MODEL_VERSION = 'acne-severity-vit', '0.1.0'
SEED, IMG_SIZE, BATCH_SIZE, EPOCHS, LR = 42, 224, 16, 8, 3e-5

# Hayashi grade -> the three severities Appendix G allows. Severe and very
# severe collapse into PRONOUNCED: the schema has no fourth level.
GRADE_TO_SEVERITY = {0: 'MILD', 1: 'MODERATE', 2: 'PRONOUNCED', 3: 'PRONOUNCED'}
CLASSES = ['MILD', 'MODERATE', 'PRONOUNCED']

# PROPOSED deployment gate. Your SRS is missing Section 5, so NFR-SAFE-008 has
# no defined numbers while FR-AI-010 gates on them. Agree these with your
# supervisor and write Section 5.
THRESHOLDS = {
    'macro_f1_min': 0.55,
    'worst_tone_accuracy_min': 0.75,
    'max_tone_accuracy_gap': 0.15,
}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('backbone:', BACKBONE_ID, '| baseline:', BASELINE_ID, '| device:', DEVICE)

In [ ]:
#@title Load labels
# ACNE04 ships `<image> <grade> <count>` label files. Some copies carry the
# grade in the filename instead (levle0_12.jpg). Both work; so does a
# labels.csv of "filename,grade" that you write yourself.
root = Path(DATA_ROOT)
images = {p.name: p for ext in ('*.jpg', '*.jpeg', '*.png') for p in root.rglob(ext)}
print(f'found {len(images)} images under {root}')
assert images, 'No images found. Check DATA_ROOT.'

rows = []
for txt in list(root.rglob('*.txt')) + list(root.rglob('*.csv')):
    for line in txt.read_text(errors='ignore').splitlines():
        parts = re.split(r'[\s,]+', line.strip())
        if len(parts) >= 2 and parts[0] in images and parts[1].isdigit():
            rows.append({'file': parts[0], 'grade': int(parts[1])})
if not rows:
    for name in images:
        m = re.search(r'l[ev]+le?(\d)', name.lower())
        if m:
            rows.append({'file': name, 'grade': int(m.group(1))})

df = pd.DataFrame(rows).drop_duplicates('file')
assert len(df), 'No labels found — add a labels.csv of "filename,grade" lines.'
df['path'] = df['file'].map(lambda f: str(images[f]))
df['severity'] = df['grade'].map(GRADE_TO_SEVERITY)
df = df.dropna(subset=['severity'])
df['y'] = df['severity'].map(CLASSES.index)
print('\nlabelled images:', len(df)); print(df['severity'].value_counts().to_string())

In [ ]:
#@title Split — stratified, fixed seed, written out
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['y'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['y'], random_state=SEED)
test_df = test_df.reset_index(drop=True)

# Saved because both models are judged on exactly these images. A metric from a
# different split is a different metric, and the comparison would be worthless.
test_df[['file', 'grade', 'severity']].to_csv(f'{OUT_DIR}/test_split.csv', index=False)
print(f'train {len(train_df)} | val {len(val_df)} | test {len(test_df)}')

In [ ]:
#@title Data loading — normalisation taken from the backbone itself
processor = AutoImageProcessor.from_pretrained(BACKBONE_ID)
NORM_MEAN = list(processor.image_mean)
NORM_STD = list(processor.image_std)
print('backbone normalisation:', NORM_MEAN, NORM_STD)

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    # Mild jitter only: heavy colour jitter shifts apparent skin tone, which is
    # the axis the fairness evaluation reports on.
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(), transforms.Normalize(NORM_MEAN, NORM_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(NORM_MEAN, NORM_STD),
])

class AcneDataset(Dataset):
    def __init__(self, frame, tf):
        self.rows, self.tf = frame.reset_index(drop=True), tf
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        row = self.rows.iloc[i]
        return self.tf(Image.open(row['path']).convert('RGB')), int(row['y'])

loaders = {
    'train': DataLoader(AcneDataset(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=2),
    'val': DataLoader(AcneDataset(val_df, eval_tf), batch_size=BATCH_SIZE, num_workers=2),
    'test': DataLoader(AcneDataset(test_df, eval_tf), batch_size=BATCH_SIZE, num_workers=2),
}
print({k: len(v.dataset) for k, v in loaders.items()})

In [ ]:
#@title OPTION A — fine-tune the Hugging Face backbone
model = AutoModelForImageClassification.from_pretrained(
    BACKBONE_ID, num_labels=len(CLASSES),
    id2label={i: c for i, c in enumerate(CLASSES)},
    label2id={c: i for i, c in enumerate(CLASSES)},
).to(DEVICE)

# Class weights: the grades are unevenly represented, and an unweighted loss
# learns to predict the majority grade while looking respectable.
counts = Counter(train_df['y'])
weights = torch.tensor([len(train_df) / (len(CLASSES) * counts[i]) for i in range(len(CLASSES))],
                       dtype=torch.float, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
optimiser = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS)

def run_epoch(loader, train: bool):
    model.train(train)
    preds, actual, total = [], [], 0.0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(pixel_values=x).logits
            loss = criterion(logits, y)
            if train:
                optimiser.zero_grad(); loss.backward(); optimiser.step()
            total += loss.item() * x.size(0)
            preds += logits.argmax(1).cpu().tolist(); actual += y.cpu().tolist()
    return total / len(loader.dataset), f1_score(actual, preds, average='macro')

best_f1, best_state = 0.0, None
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1 = run_epoch(loaders['train'], True)
    _, va_f1 = run_epoch(loaders['val'], False)
    scheduler.step()
    star = ''
    if va_f1 > best_f1:
        best_f1 = va_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        star = '  <- best'
    print(f'epoch {epoch:2d}  loss {tr_loss:.3f}  train_f1 {tr_f1:.3f}  val_f1 {va_f1:.3f}{star}')

model.load_state_dict(best_state)
print(f'\nbest validation macro-F1: {best_f1:.3f}')

In [ ]:
#@title Evaluate OUR model on the held-out test split
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in loaders['test']:
        y_pred += model(pixel_values=x.to(DEVICE)).logits.argmax(1).cpu().tolist()
        y_true += y.tolist()
y_true, y_pred = np.array(y_true), np.array(y_pred)

print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3, zero_division=0))
print('confusion matrix (rows = actual, cols = predicted)')
print(pd.DataFrame(confusion_matrix(y_true, y_pred), index=CLASSES, columns=CLASSES).to_string())

ours_accuracy = float((y_true == y_pred).mean())
ours_macro_f1 = float(f1_score(y_true, y_pred, average='macro'))
print(f'\nOURS: accuracy {ours_accuracy:.3f} | macro-F1 {ours_macro_f1:.3f}')
print('Specificity is not measurable here: ACNE04 contains no acne-free faces.')

In [ ]:
#@title OPTION B — the public pretrained model, same images
# Runs locally, so no dataset image is sent anywhere. That matters: the ACNE04
# licence may well forbid sharing images with a third-party API, which is why
# this comparison is against a downloadable model rather than a hosted one.
base_processor = AutoImageProcessor.from_pretrained(BASELINE_ID)
base_model = AutoModelForImageClassification.from_pretrained(BASELINE_ID).to(DEVICE).eval()
base_labels = base_model.config.id2label
print('baseline labels:', base_labels)

def map_baseline_label(name: str):
    """Fold the baseline's own grades onto our three severities.

    The two label sets were not designed together, so this mapping is a
    judgement and is printed in full for the report. 'clear' and 'occasional'
    fold into MILD because ACNE04 has no acne-free faces -- the baseline
    predicting 'clear' on an image that does have acne is an under-call, and
    MILD is the closest our scale can represent.
    """
    n = name.lower()
    if 'very' in n and 'severe' in n: return 'PRONOUNCED'
    if 'severe' in n:                 return 'PRONOUNCED'
    if 'moderate' in n:               return 'MODERATE'
    if 'mild' in n:                   return 'MILD'
    if 'clear' in n or 'occasional' in n: return 'MILD'
    return None

mapping = {i: map_baseline_label(n) for i, n in base_labels.items()}
print('\nlabel mapping used (put this in your report):')
for i, n in base_labels.items():
    print(f'  {n:>18}  ->  {mapping[i]}')
unmapped = [n for i, n in base_labels.items() if mapping[i] is None]
if unmapped:
    print('UNMAPPED (counted as wrong):', unmapped)

base_pred = []
with torch.no_grad():
    for start in range(0, len(test_df), BATCH_SIZE):
        batch = test_df.iloc[start:start + BATCH_SIZE]
        imgs = [Image.open(p).convert('RGB') for p in batch['path']]
        inputs = base_processor(images=imgs, return_tensors='pt').to(DEVICE)
        ids = base_model(**inputs).logits.argmax(1).cpu().tolist()
        base_pred += [mapping.get(i) for i in ids]

# An unmapped prediction cannot be scored as correct; -1 never matches a class.
base_y_pred = np.array([CLASSES.index(s) if s in CLASSES else -1 for s in base_pred])
base_accuracy = float((y_true == base_y_pred).mean())
base_macro_f1 = float(f1_score(y_true, base_y_pred, average='macro', labels=range(len(CLASSES)), zero_division=0))
print(f'\nBASELINE: accuracy {base_accuracy:.3f} | macro-F1 {base_macro_f1:.3f}')
print(pd.DataFrame(confusion_matrix(y_true, base_y_pred, labels=range(len(CLASSES))),
                   index=CLASSES, columns=CLASSES).to_string())

In [ ]:
#@title Skin-tone breakdown for BOTH models (NFR-SAFE-008)
# ACNE04 has no skin-tone labels, so tone is ESTIMATED from each image with ITA
# (Individual Typology Angle) over CIELAB. An estimate, and the output says so --
# but it is the only way to produce the breakdown NFR-SAFE-008 requires from a
# dataset that lacks the labels (OI-016).
from skimage import color as skcolor

def estimate_ita(path: str) -> float:
    img = np.asarray(Image.open(path).convert('RGB').resize((128, 128))) / 255.0
    h, w, _ = img.shape
    centre = img[h // 4: 3 * h // 4, w // 4: 3 * w // 4]  # mostly face, least hair/background
    lab = skcolor.rgb2lab(centre)
    L, b = lab[..., 0], lab[..., 2]
    # Drop blown highlights and deep shadow, which skew ITA. The window is wide
    # on purpose: a narrow one (L < 90) discarded every pixel of very fair skin
    # in testing, which would have quietly dropped the lightest group out of the
    # fairness breakdown -- the opposite of what NFR-SAFE-008 is for.
    keep = (L > 15) & (L < 97)
    if keep.sum() < 50:
        keep = np.ones_like(L, dtype=bool)
    return float(np.degrees(np.arctan2(np.median(L[keep]) - 50, np.median(b[keep]))))

def ita_group(ita: float) -> str:
    if np.isnan(ita): return 'unknown'
    if ita > 55: return 'very light'
    if ita > 41: return 'light'
    if ita > 28: return 'intermediate'
    if ita > 10: return 'tan'
    if ita > -30: return 'brown'
    return 'dark'

tone_df = test_df.copy()
tone_df['ita'] = [estimate_ita(p) for p in tone_df['path']]
tone_df['tone'] = tone_df['ita'].map(ita_group)
tone_df['ours_correct'] = (y_true == y_pred)
tone_df['baseline_correct'] = (y_true == base_y_pred)

by_tone = (tone_df.groupby('tone')
           .agg(images=('ours_correct', 'size'),
                ours=('ours_correct', 'mean'),
                baseline=('baseline_correct', 'mean'),
                median_ita=('ita', 'median'))
           .sort_values('median_ita', ascending=False))
print('accuracy by estimated skin tone\n')
print(by_tone.round(3).to_string())

# Groups too small to mean anything are excluded from the gate, and the
# exclusion is printed rather than hidden.
MIN_GROUP = 15
usable = by_tone[(by_tone['images'] >= MIN_GROUP) & (by_tone.index != 'unknown')]
print(f'\ngroups with >= {MIN_GROUP} images: {list(usable.index) or "NONE"}')
if len(usable) < 2:
    print('WARNING: too few tone groups to judge fairness. Report as a limitation.')
worst_tone_acc = float(usable['ours'].min()) if len(usable) else None
tone_gap = float(usable['ours'].max() - usable['ours'].min()) if len(usable) > 1 else None

In [ ]:
#@title HEAD-TO-HEAD table (for the report)
comparison = pd.DataFrame([
    {'model': f'Ours — {BACKBONE_ID} fine-tuned', 'accuracy': round(ours_accuracy, 3),
     'macro_f1': round(ours_macro_f1, 3),
     'trained_on': 'ACNE04 (licence recorded)', 'training_data_known': 'yes'},
    {'model': f'Baseline — {BASELINE_ID}', 'accuracy': round(base_accuracy, 3),
     'macro_f1': round(base_macro_f1, 3),
     'trained_on': 'not disclosed', 'training_data_known': 'no'},
]).set_index('model')
print(comparison.to_string())

print('\nper skin tone (accuracy)')
print(by_tone[['images', 'ours', 'baseline']].round(3).to_string())

print('\nRead this fairly in your report:')
print('- Both were scored on the SAME held-out images.')
print('- Ours was fine-tuned on ACNE04, so ACNE04 is home turf for it and away for the baseline.')
print('- The baseline uses its own severity scale, folded onto ours by the printed mapping.')
print('- Losing to the baseline would still be a result worth reporting.')
comparison.to_csv(f'{OUT_DIR}/comparison.csv')
by_tone.to_csv(f'{OUT_DIR}/by_skin_tone.csv')

In [ ]:
#@title Export to ONNX and time it on CPU (SRS 2.4, NFR-PERF-001)
class LogitsOnly(nn.Module):
    """ONNX wants a tensor out; a Hugging Face model returns an object."""
    def __init__(self, inner):
        super().__init__(); self.inner = inner
    def forward(self, pixel_values):
        return self.inner(pixel_values=pixel_values).logits

onnx_path = f'{OUT_DIR}/{MODEL_ID}-{MODEL_VERSION}.onnx'
exportable = LogitsOnly(model.cpu().eval()).eval()
torch.onnx.export(exportable, torch.randn(1, 3, IMG_SIZE, IMG_SIZE), onnx_path,
                  input_names=['image'], output_names=['logits'],
                  dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}}, opset_version=17)

import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])

# The exported file must agree with the model that was measured, or the metrics
# above describe something other than what would be deployed.
sample, _ = loaders['test'].dataset[0]
torch_out = exportable(sample.unsqueeze(0)).detach().numpy()
onnx_out = sess.run(None, {'image': sample.unsqueeze(0).numpy()})[0]
agree = bool(np.allclose(torch_out, onnx_out, atol=1e-3))

times = []
for _ in range(20):
    t0 = time.perf_counter(); sess.run(None, {'image': sample.unsqueeze(0).numpy()}); times.append(time.perf_counter() - t0)
cpu_ms = float(np.median(times) * 1000)
sha256 = hashlib.sha256(Path(onnx_path).read_bytes()).hexdigest()

print(f'exported {onnx_path}')
print(f'size {Path(onnx_path).stat().st_size / 1e6:.1f} MB | sha256 {sha256[:16]}...')
print(f'pytorch == onnx: {agree} | median CPU inference {cpu_ms:.0f} ms')
if cpu_ms > 2000:
    print('NOTE: slow on CPU. Production runs CPU-only (SRS 2.4) — factor this into NFR-PERF-001.')

In [ ]:
#@title Deployment gate (FR-AI-010) and Appendix I entry
checks = {
    'macro_f1': (ours_macro_f1, THRESHOLDS['macro_f1_min'], ours_macro_f1 >= THRESHOLDS['macro_f1_min']),
    'worst_tone_accuracy': (worst_tone_acc, THRESHOLDS['worst_tone_accuracy_min'],
                            worst_tone_acc is not None and worst_tone_acc >= THRESHOLDS['worst_tone_accuracy_min']),
    'tone_gap': (tone_gap, THRESHOLDS['max_tone_accuracy_gap'],
                 tone_gap is not None and tone_gap <= THRESHOLDS['max_tone_accuracy_gap']),
    'beats_public_baseline': (ours_macro_f1, base_macro_f1, ours_macro_f1 > base_macro_f1),
    'onnx_matches_pytorch': (agree, True, agree),
}
print('DEPLOYMENT GATE (FR-AI-010)')
for name, (value, limit, ok) in checks.items():
    shown = f'{value:.3f}' if isinstance(value, float) else str(value)
    lim = f'{limit:.3f}' if isinstance(limit, float) else str(limit)
    print(f'  {"PASS" if ok else "FAIL"}  {name:24} {shown:>8}  (limit {lim})')

passed = all(ok for _, _, ok in checks.values())
print(f'\nVERDICT: {"PASSES the measurable checks" if passed else "DOES NOT PASS"}')
print('Specificity remains UNMEASURED: ACNE04 has no acne-free faces, so how often\n'
      'either model invents acne on clear skin is unknown. State this as a limitation.')

entry = {
    'model': {
        'modelId': MODEL_ID, 'version': MODEL_VERSION, 'task': 'ACNE_DETECTION',
        'architecture': f'{BACKBONE_ID} fine-tuned, 3-class severity head',
        'backboneLicence': 'Apache-2.0 (google/vit-base-patch16-224-in21k)',
        'artefact': Path(onnx_path).name, 'sha256': sha256,
        'inputSize': IMG_SIZE, 'normalisation': {'mean': NORM_MEAN, 'std': NORM_STD},
        'classes': CLASSES, 'cpuInferenceMs': round(cpu_ms),
        'evaluatedAt': time.strftime('%Y-%m-%d'), 'deploymentStatus': 'NOT_DEPLOYED',
    },
    'dataset': {
        'name': 'ACNE04', 'source': 'FILL IN: URL you downloaded it from',
        'licence': 'FILL IN: licence name/text', 'licencePermitsUse': 'FILL IN: yes/no',
        'imageFormat': 'face photographs',
        'population': 'FILL IN: as described by the authors',
        'labelProvenance': 'Hayashi grading by the dataset authors',
        'knownLabelIssues': 'Grades are subjective; population differs from target market (ASM-005, OI-018)',
        'images': int(len(df)), 'trainValTest': [len(train_df), len(val_df), len(test_df)],
    },
    'evaluation': {
        'accuracy': round(ours_accuracy, 3), 'macroF1': round(ours_macro_f1, 3),
        'perClass': classification_report(y_true, y_pred, target_names=CLASSES,
                                          output_dict=True, zero_division=0),
        'bySkinTone': json.loads(by_tone.round(3).reset_index().to_json(orient='records')),
        'skinToneMethod': 'ITA estimated from image (dataset has no labels) — approximate',
        'specificity': None,
        'specificityNote': 'Not measurable: ACNE04 contains no acne-free faces',
        'thresholds': THRESHOLDS, 'thresholdsStatus': 'PROPOSED — SRS Section 5 is missing',
        'gatePassed': bool(passed),
    },
    'baselineComparison': {
        'modelId': BASELINE_ID, 'licence': 'MIT', 'trainingDataDisclosed': False,
        'accuracy': round(base_accuracy, 3), 'macroF1': round(base_macro_f1, 3),
        'labelMapping': {str(base_labels[i]): mapping[i] for i in base_labels},
        'note': 'Same test images. ACNE04 is in-domain for ours and out-of-domain for the baseline.',
    },
}
print('\n' + '=' * 70)
print('COPY EVERYTHING BELOW AND PASTE IT BACK')
print('=' * 70)
print(json.dumps(entry, indent=2))
Path(f'{OUT_DIR}/appendix_i_entry.json').write_text(json.dumps(entry, indent=2))